# DramValue Whisky Price Analysis

This notebook contains whisky secondary market price data from DramValue.com.

**Data Sources:**
- Scotch Whisky Auctions (auction hammer prices)
- Whisky Auctioneer (auction hammer prices)
- Dekanta (retail prices)
- The Whisky Barrel (retail prices)

**Dataset Features:**
- Bottle information (name, distillery, age, proof, etc.)
- Price data (USD, original currency)
- Source and transaction dates
- Market statistics (avg, min, max prices)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Load the data
df = pd.read_csv('dramvalue_prices.csv')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Data overview
print("\n=== Dataset Info ===")
print(f"Total price records: {len(df):,}")
print(f"Unique bottles: {df['bottle_id'].nunique():,}")
print(f"Date range: {df['transaction_date'].min()} to {df['transaction_date'].max()}")
print(f"\n=== Price Statistics ===")
print(df['price_usd'].describe())

In [ ]:
# Data by source
print("\n=== Data by Source ===")
source_stats = df.groupby('source_name').agg({
    'price_id': 'count',
    'price_usd': ['mean', 'median', 'min', 'max']
}).round(2)
source_stats.columns = ['count', 'avg_price', 'median_price', 'min_price', 'max_price']
source_stats

In [ ]:
# Category distribution
print("\n=== Category Distribution ===")
category_counts = df['category'].value_counts()
print(category_counts)

fig, ax = plt.subplots(figsize=(10, 6))
category_counts.plot(kind='bar', ax=ax, color='amber')
ax.set_title('Price Records by Category')
ax.set_xlabel('Category')
ax.set_ylabel('Number of Records')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All prices
axes[0].hist(df['price_usd'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Price Distribution (All)')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Frequency')

# Prices under $1000 (more detail)
under_1000 = df[df['price_usd'] < 1000]['price_usd']
axes[1].hist(under_1000.dropna(), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('Price Distribution (Under $1,000)')
axes[1].set_xlabel('Price (USD)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Top distilleries by number of sales
print("\n=== Top 20 Distilleries by Sales Volume ===")
top_distilleries = df.groupby('distillery').agg({
    'price_id': 'count',
    'price_usd': 'mean'
}).sort_values('price_id', ascending=False).head(20)
top_distilleries.columns = ['sales_count', 'avg_price']
top_distilleries['avg_price'] = top_distilleries['avg_price'].round(2)
top_distilleries

In [ ]:
# Most expensive bottles sold
print("\n=== Top 20 Most Expensive Sales ===")
expensive = df.nlargest(20, 'price_usd')[['bottle_name', 'distillery', 'price_usd', 'source_name', 'transaction_date']]
expensive

In [ ]:
# Price by age statement
df_with_age = df[df['age_statement'].notna() & (df['age_statement'] > 0)].copy()
if len(df_with_age) > 0:
    age_prices = df_with_age.groupby('age_statement')['price_usd'].agg(['mean', 'count']).reset_index()
    age_prices = age_prices[age_prices['count'] >= 5]  # At least 5 samples
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(age_prices['age_statement'], age_prices['mean'], color='brown', alpha=0.7)
    ax.set_title('Average Price by Age Statement')
    ax.set_xlabel('Age (Years)')
    ax.set_ylabel('Average Price (USD)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Auction vs Retail price comparison
source_comparison = df.groupby('source_type')['price_usd'].agg(['mean', 'median', 'count']).round(2)
print("\n=== Auction vs Retail Prices ===")
print(source_comparison)

In [ ]:
# Time series of prices (if transaction dates available)
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')
df_dated = df.dropna(subset=['transaction_date'])

if len(df_dated) > 0:
    monthly = df_dated.set_index('transaction_date').resample('M')['price_usd'].agg(['mean', 'count'])
    monthly = monthly[monthly['count'] >= 10]  # At least 10 sales per month
    
    if len(monthly) > 1:
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.plot(monthly.index, monthly['mean'], marker='o', linewidth=2)
        ax.set_title('Average Whisky Price Over Time')
        ax.set_xlabel('Date')
        ax.set_ylabel('Average Price (USD)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

In [ ]:
# Export summary statistics
summary = {
    'total_records': len(df),
    'unique_bottles': df['bottle_id'].nunique(),
    'unique_distilleries': df['distillery'].nunique(),
    'avg_price_usd': df['price_usd'].mean().round(2),
    'median_price_usd': df['price_usd'].median().round(2),
    'min_price_usd': df['price_usd'].min(),
    'max_price_usd': df['price_usd'].max(),
}
print("\n=== Dataset Summary ===")
for k, v in summary.items():
    print(f"{k}: {v:,}" if isinstance(v, int) else f"{k}: ${v:,.2f}" if 'price' in k else f"{k}: {v}")

## Data Dictionary

| Column | Description |
|--------|-------------|
| bottle_id | Unique identifier for the bottle |
| bottle_name | Full name of the bottle |
| distillery | Distillery/producer name |
| brand | Brand name |
| category | Spirit category (SCOTCH_SINGLE_MALT, BOURBON, etc.) |
| age_statement | Age in years (if stated) |
| proof | Alcohol proof |
| size_ml | Bottle size in milliliters |
| release_year | Year of release |
| is_limited_release | Whether it's a limited release (t/f) |
| is_allocated | Whether it's an allocated bottle (t/f) |
| msrp | Manufacturer's suggested retail price |
| avg_price | Average historical price for this bottle |
| min_price | Minimum recorded price for this bottle |
| max_price | Maximum recorded price for this bottle |
| price_id | Unique identifier for this price record |
| price_usd | Price in USD |
| currency | Original currency of the sale (USD, GBP, EUR, JPY) |
| source_type | Source type (AUCTION or RETAIL) |
| source_name | Name of the source (e.g., Scotch Whisky Auctions) |
| source_url | URL of the original listing |
| transaction_date | Date of the sale/listing |
| scraped_at | When the data was collected |

## Bottles Dataset (dramvalue_bottles.csv)

| Column | Description |
|--------|-------------|
| id | Unique bottle identifier |
| name | Full bottle name |
| distillery | Distillery name |
| brand | Brand name |
| category | Spirit category |
| age_statement | Age in years |
| proof | Alcohol proof |
| size_ml | Bottle size in ml |
| release_year | Year released |
| is_limited_release | Limited release flag |
| is_allocated | Allocated bottle flag |
| msrp | MSRP if known |
| avg_price | Average market price |
| min_price | Lowest recorded price |
| max_price | Highest recorded price |
| last_price | Most recent price |
| last_price_date | Date of most recent price |
| price_count | Number of price records |
| price_trend | Price trend indicator |
| created_at | Record creation date |
| updated_at | Last update date |